# StudyHabit: Social-Science Survey Analysis
## Khảo sát thói quen học tập của sinh viên

This Jupyter Notebook provides exploratory data analysis (EDA), descriptive statistics, and visualization for the university research project **StudyHabit**.

### Pipeline Overview:
1. **Ingestion:** Load exported responses from Google Sheets / CSV.
2. **Data Cleaning:** Unpack JSON responses, validate completeness, impute missing values.
3. **Descriptive Statistics:** Calculate mean study hours, concentration, planning, and sleep impacts.
4. **Visualization:** Plot distribution of study times, preferred locations, learning methods, and correlation with effectiveness.

In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plot aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']

csv_path = 'sample_responses.csv'
raw_df = pd.read_csv(csv_path)
print(f"Loaded {len(raw_df)} survey records from Google Sheets export.")
raw_df.head(2)

### 2. Parse JSON Answers and Clean Data

In [2]:
parsed_rows = []
for idx, row in raw_df.iterrows():
    ans_raw = row.get('answers', '{}')
    ans = json.loads(ans_raw) if isinstance(ans_raw, str) else ans_raw
    parsed_rows.append({
        'id': row.get('id'),
        'year': ans.get('sh-q1'),
        'major': ans.get('sh-q2'),
        'study_time': ans.get('sh-q3'),
        'days_per_week': ans.get('sh-q4'),
        'time_of_day': ans.get('sh-q5'),
        'location': ans.get('sh-q6'),
        'format': ans.get('sh-q7'),
        'learning_methods': ans.get('sh-q8', []),
        'concentration': pd.to_numeric(ans.get('sh-q11'), errors='coerce'),
        'social_media': pd.to_numeric(ans.get('sh-q12'), errors='coerce'),
        'device': ans.get('sh-q13'),
        'ai_usage': ans.get('sh-q14'),
        'sleep': ans.get('sh-q15'),
        'effectiveness': pd.to_numeric(ans.get('sh-q17'), errors='coerce')
    })

df = pd.DataFrame(parsed_rows)
df.info()

### 3. Summary Statistics

In [3]:
summary = {
    'Total Respondents': len(df),
    'Mean Concentration (1-5)': round(df['concentration'].mean(), 2),
    'Mean Perceived Effectiveness (1-5)': round(df['effectiveness'].mean(), 2),
    'Mean Social Media Distraction (1-5)': round(df['social_media'].mean(), 2)
}
pd.Series(summary)

### 4. Visualizations: Study Hours & Learning Methods

In [4]:
plt.figure(figsize=(8, 4.5))
order = ['Dưới 1 giờ', '1–2 giờ', '2–4 giờ', '4–6 giờ', 'Trên 6 giờ']
counts = df['study_time'].value_counts().reindex(order).fillna(0)
bars = plt.bar(order, counts, color='#0284c7', width=0.55)
plt.title('Daily Independent Study Time (Hours/Day)', fontweight='bold')
plt.ylabel('Students')
plt.show()

### 5. Correlation: Study Duration vs. Perceived Effectiveness

In [5]:
eff_by_time = df.groupby('study_time')['effectiveness'].mean().reindex(order)
plt.figure(figsize=(7, 4))
plt.plot(eff_by_time.index, eff_by_time.values, marker='o', color='#0369a1', linewidth=2)
plt.title('Study Duration vs. Effectiveness Rating', fontweight='bold')
plt.ylabel('Average Rating (1–5)')
plt.ylim(1.0, 5.0)
plt.show()